# Probe testing

All setup, model/dataset classes, and eval helpers live in `probe_lib.py` next to this
notebook -- this notebook is just the test schedule itself. Edit `probe_lib.py` in an editor;
`%autoreload` picks up changes without restarting the kernel.

Run the setup cell below once per session, then run whichever test cells you need. Every test
cell is self-contained (loads its own checkpoint, builds its own eval set) and ends by printing
the updated comparison table.

In [2]:
%load_ext autoreload
%autoreload 2

from probe_lib import *

odf = load_and_prepare_data("../may20/full_babe_with_cats.csv")
cat_cardinalities, cat_embs_info, out_cats, in_cats = build_cat_embs_info()

train_df, holdout_df = get_train_test_split(odf)

print(device)
print(f"odf: {len(odf)} rows | train: {len(train_df)} | holdout: {len(holdout_df)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/ihome/xli/sek188/MSTHESIS/jul_26/probe_lib.py:84: DtypeWarning: Columns (0: survey_completed) have mixed types. Specify dtype option on import or set low_memory=False.
  odf = pd.read_csv(csv_path)


cuda
odf: 49734 rows | train: 39787 | holdout: 9947


## GO-NO-GO sanity check

Loads the baseline checkpoint and evaluates it on the hold-out set three ways: real labels/targets,
shuffled labels, and shuffled target categories. If shuffling doesn't tank macro F1 back down
toward the marginal/baseline rate, something in the harness is wrong (e.g. the shuffle isn't
taking effect, or metrics are reading the wrong columns) -- fix that before trusting any test below.

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

sanity_checks = [
    ("real_labels_real_cats", holdout_df),
    ("shuffled_labels", shuffle_labels(holdout_df)),
    ("shuffled_target_cats", shuffle_target_cats(holdout_df, out_cats)),
]

for check_name, check_df in sanity_checks:
    eval_dataset = MTLDataset(check_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"go_no_go__{check_name}",
        batch_size=64,
        device=device,
        entropy_top_fracs=[],  # skip entropy breakdown, this is just a quick sanity pass
    )

    print_eval_report(result, title=check_name)

## Test 1 -- TinyLlama, condEntropyLoss, 3 epoch, no modification

Covers "eval on hold-out set", "eval on full set", and "eval on high-entropy rows (50/25/10%)"
in one cell, since they all reuse the same loaded checkpoint and cached predictions.

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"
CONDITION = "noShuffle"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_scopes = [
    ("hold_out", holdout_df),
    ("full", odf),
]

for scope_name, scope_df in eval_scopes:
    eval_dataset = MTLDataset(scope_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"{CHECKPOINT_NAME}__{CONDITION}__{scope_name}",
        batch_size=64,
        device=device,
        refresh_predictions=REFRESH_PREDICTIONS,
    )

    print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | {scope_name}")

    rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope=scope_name)
    append_and_show_results(rows)

    write_eval_log(
        result,
        path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__{scope_name}_log.json",
        model_name=CHECKPOINT_NAME,
        condition=CONDITION,
        extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": scope_name, "n_eval_rows": len(scope_df)},
    )

## Test 2 -- TinyLlama, condEntropyLoss, 3 epoch, zeroCats

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"
CONDITION = "zeroCats"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_scopes = [
    ("hold_out", holdout_df),
    ("full", odf),
]

for scope_name, base_df in eval_scopes:
    scope_df = zero_target_cats(base_df, out_cats)
    eval_dataset = MTLDataset(scope_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"{CHECKPOINT_NAME}__{CONDITION}__{scope_name}",
        batch_size=64,
        device=device,
        refresh_predictions=REFRESH_PREDICTIONS,
    )

    print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | {scope_name}")

    rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope=scope_name)
    append_and_show_results(rows)

    write_eval_log(
        result,
        path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__{scope_name}_log.json",
        model_name=CHECKPOINT_NAME,
        condition=CONDITION,
        extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": scope_name, "n_eval_rows": len(scope_df)},
    )

## Test 3 -- TinyLlama, condEntropyLoss, 3 epoch, zeroLabs

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"
CONDITION = "zeroLabs"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_scopes = [
    ("hold_out", holdout_df),
    ("full", odf),
]

for scope_name, base_df in eval_scopes:
    scope_df = zero_labels(base_df)
    eval_dataset = MTLDataset(scope_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"{CHECKPOINT_NAME}__{CONDITION}__{scope_name}",
        batch_size=64,
        device=device,
        refresh_predictions=REFRESH_PREDICTIONS,
    )

    print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | {scope_name}")

    rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope=scope_name)
    append_and_show_results(rows)

    write_eval_log(
        result,
        path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__{scope_name}_log.json",
        model_name=CHECKPOINT_NAME,
        condition=CONDITION,
        extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": scope_name, "n_eval_rows": len(scope_df)},
    )

## Test 4 -- TinyLlama, condEntropyLoss, 3 epoch, shuffleCats

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"
CONDITION = "shuffleCats"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_scopes = [
    ("hold_out", holdout_df),
    ("full", odf),
]

for scope_name, base_df in eval_scopes:
    scope_df = shuffle_target_cats(base_df, out_cats)
    eval_dataset = MTLDataset(scope_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"{CHECKPOINT_NAME}__{CONDITION}__{scope_name}",
        batch_size=64,
        device=device,
        refresh_predictions=REFRESH_PREDICTIONS,
    )

    print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | {scope_name}")

    rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope=scope_name)
    append_and_show_results(rows)

    write_eval_log(
        result,
        path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__{scope_name}_log.json",
        model_name=CHECKPOINT_NAME,
        condition=CONDITION,
        extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": scope_name, "n_eval_rows": len(scope_df)},
    )

## Test 5 -- TinyLlama, condEntropyLoss, 3 epoch, shuffleLabs

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_standard"
CONDITION = "shuffleLabs"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_scopes = [
    ("hold_out", holdout_df),
    ("full", odf),
]

for scope_name, base_df in eval_scopes:
    scope_df = shuffle_labels(base_df)
    eval_dataset = MTLDataset(scope_df, cat_embs_info, tokenizer, max_length=128)

    result = evaluate_raw(
        probe,
        eval_dataset,
        test_name=f"{CHECKPOINT_NAME}__{CONDITION}__{scope_name}",
        batch_size=64,
        device=device,
        refresh_predictions=REFRESH_PREDICTIONS,
    )

    print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | {scope_name}")

    rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope=scope_name)
    append_and_show_results(rows)

    write_eval_log(
        result,
        path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__{scope_name}_log.json",
        model_name=CHECKPOINT_NAME,
        condition=CONDITION,
        extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": scope_name, "n_eval_rows": len(scope_df)},
    )

## Test 6 -- TinyLlama, condEntropyLoss, 5 epoch, no modification, hold-out only

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_5epoch_standard"
CONDITION = "noShuffle"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_dataset = MTLDataset(holdout_df, cat_embs_info, tokenizer, max_length=128)

result = evaluate_raw(
    probe,
    eval_dataset,
    test_name=f"{CHECKPOINT_NAME}__{CONDITION}__hold_out",
    batch_size=64,
    device=device,
    refresh_predictions=REFRESH_PREDICTIONS,
)

print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | hold_out")

rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope="hold_out")
append_and_show_results(rows)

write_eval_log(
    result,
    path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__hold_out_log.json",
    model_name=CHECKPOINT_NAME,
    condition=CONDITION,
    extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": "hold_out", "n_eval_rows": len(holdout_df)},
)

## Test 7 -- TinyLlama, condEntropyLoss, 10 epoch, no modification, hold-out only

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_10epoch_standard"
CONDITION = "noShuffle"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_dataset = MTLDataset(holdout_df, cat_embs_info, tokenizer, max_length=128)

result = evaluate_raw(
    probe,
    eval_dataset,
    test_name=f"{CHECKPOINT_NAME}__{CONDITION}__hold_out",
    batch_size=64,
    device=device,
    refresh_predictions=REFRESH_PREDICTIONS,
)

print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | hold_out")

rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope="hold_out")
append_and_show_results(rows)

write_eval_log(
    result,
    path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__hold_out_log.json",
    model_name=CHECKPOINT_NAME,
    condition=CONDITION,
    extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": "hold_out", "n_eval_rows": len(holdout_df)},
)

## Test 8 -- TinyLlama, standardLoss (no conditional-entropy weighting), 3 epoch, hold-out only

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
CHECKPOINT_NAME = "TinyLlama_probe_3epoch_stdLoss"
CONDITION = "noShuffle"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_dataset = MTLDataset(holdout_df, cat_embs_info, tokenizer, max_length=128)

result = evaluate_raw(
    probe,
    eval_dataset,
    test_name=f"{CHECKPOINT_NAME}__{CONDITION}__hold_out",
    batch_size=64,
    device=device,
    refresh_predictions=REFRESH_PREDICTIONS,
)

print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | hold_out")

rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope="hold_out")
append_and_show_results(rows)

write_eval_log(
    result,
    path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__hold_out_log.json",
    model_name=CHECKPOINT_NAME,
    condition=CONDITION,
    extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": "hold_out", "n_eval_rows": len(holdout_df)},
)

## Test 9 -- Qwen, condEntropyLoss, 5 epoch, hold-out only

`QWEN_MODEL_NAME` below is a placeholder -- set it to whatever exact HF model id the training
doc actually used.

In [ ]:
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # TODO: confirm against the training doc
CHECKPOINT_NAME = "Qwen_probe_5epoch_standard"
CONDITION = "noShuffle"
REFRESH_PREDICTIONS = False

checkpoint_path = os.path.join(PROBE_WEIGHTS_DIR, CHECKPOINT_NAME, "model.safetensors")

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

parameters = {"condEntropyWeight": 1.0}
probe = ambiProbe(cat_embs_info, QWEN_MODEL_NAME, parameters)
probe.load_state_dict(load_file(checkpoint_path))
probe.to(device)
probe.eval()

eval_dataset = MTLDataset(holdout_df, cat_embs_info, tokenizer, max_length=128)

result = evaluate_raw(
    probe,
    eval_dataset,
    test_name=f"{CHECKPOINT_NAME}__{CONDITION}__hold_out",
    batch_size=64,
    device=device,
    refresh_predictions=REFRESH_PREDICTIONS,
)

print_eval_report(result, title=f"{CHECKPOINT_NAME} | {CONDITION} | hold_out")

rows = build_result_rows(result, model_name=CHECKPOINT_NAME, condition=CONDITION, eval_scope="hold_out")
append_and_show_results(rows)

write_eval_log(
    result,
    path=f"tests/{CHECKPOINT_NAME}/{CONDITION}__hold_out_log.json",
    model_name=CHECKPOINT_NAME,
    condition=CONDITION,
    extra={"batch_size": 64, "seed": RANDOM_SEED, "eval_scope": "hold_out", "n_eval_rows": len(holdout_df)},
)

## Final comparison table

Every run appended a row (or several, one per entropy fraction) to `giant_res.csv`. Full table:

In [ ]:
giant_res = pd.read_csv(GIANT_RESULTS_DICT)

display_cols = [
    "model", "condition", "eval_scope", "entropy_top_frac", "n_rows",
    "accuracy", "baseline_accuracy", "accuracy_lift",
    "macro_f1", "baseline_macro_f1", "macro_f1_lift",
]
display_cols = [c for c in display_cols if c in giant_res.columns]

giant_res[display_cols].sort_values(["model", "condition", "eval_scope", "entropy_top_frac"])